# FGSM Adversarial Attack Demo

This notebook shows how **one-step gradient-based attacks** (Fast Gradient Sign Method, FGSM) can change a pretrained **ResNet-50** ImageNet classifier’s prediction by adding a tiny, often **human-imperceptible** perturbation to the input image.

Run on **GPU** (Runtime → Change runtime type → GPU) for a quick interactive demo.

## Run it fast
1. Switch runtime to **GPU**.
2. Run all cells.
3. (Optional) Change the image URL (defaults to a traffic sign).
4. Change `epsilons` and re-run the attack + plots.

In [ ]:
import io
import requests
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision import models
from torchvision.models import ResNet50_Weights
import matplotlib.pyplot as plt
from PIL import Image

# Device: tensors are moved to GPU on Colab when available (batch dim N = 1 for single image)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Default demo image: traffic sign photo (good self-driving analogy; also in ImageNet classes)
URL = "https://upload.wikimedia.org/wikipedia/commons/8/8a/Stop_Sign_China.jpg"
# Alternative (very reliable): ImageNet-style animal photo from PyTorch Hub assets
# URL = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"
resp = requests.get(URL, timeout=30)
resp.raise_for_status()
pil_img = Image.open(io.BytesIO(resp.content)).convert("RGB")

# Float tensor in [0, 1], shape (1, 3, 224, 224) — NCHW for the classifier
resize = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor()])
img_tensor = resize(pil_img).unsqueeze(0).to(device)

# ImageNet mean/std broadcast to (1, 3, 1, 1) for (N, 3, H, W) batches
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)


def imagenet_normalize(x):
    # x: (N, 3, H, W) in [0, 1]; output same shape, normalized for ResNet
    return (x - IMAGENET_MEAN) / IMAGENET_STD


weights = ResNet50_Weights.IMAGENET1K_V1
model = models.resnet50(weights=weights).to(device).eval()
categories = weights.meta["categories"]

In [ ]:
def fgsm_attack(image, epsilon, data_grad):
    # image: (N, 3, H, W); data_grad: same shape as image (gradient w.r.t. input)
    sign_data_grad = data_grad.sign()
    perturbed_image = image + epsilon * sign_data_grad
    perturbed_image = torch.clamp(perturbed_image, 0, 1)
    return perturbed_image

In [ ]:
with torch.no_grad():
    logits = model(imagenet_normalize(img_tensor))  # logits: (1, num_classes)
    probs = torch.softmax(logits, dim=1).squeeze(0)  # (num_classes,)

top5_prob, top5_idx = probs.topk(5)
top5_prob = top5_prob.cpu().numpy()
top5_idx = top5_idx.cpu().numpy()
labels = [categories[i] for i in top5_idx]

fig, ax = plt.subplots(figsize=(8, 4))
y = np.arange(5)
ax.barh(y, top5_prob, color="steelblue")
ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlabel("Confidence")
ax.set_title("Top-5 predictions (original image)")
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
epsilons = [0.0, 0.005, 0.01, 0.02, 0.05, 0.1, 0.3]
attack_results = []

# Original top-1 index (treated as the "correct" class for measuring robustness)
with torch.no_grad():
    orig_logits = model(imagenet_normalize(img_tensor))  # (1, num_classes)
orig_label = orig_logits.argmax(dim=1).item()  # scalar index

for eps in epsilons:
    if eps == 0.0:
        adv = img_tensor.clone().detach()  # (1, 3, 224, 224)
    else:
        x = img_tensor.clone().detach().requires_grad_(True)  # (1, 3, 224, 224)
        logits = model(imagenet_normalize(x))  # (1, num_classes)
        target = torch.tensor([orig_label], device=device)  # (1,)
        loss = nn.CrossEntropyLoss()(logits, target)  # scalar
        model.zero_grad(set_to_none=True)
        if x.grad is not None:
            x.grad.zero_()
        loss.backward()
        adv = fgsm_attack(x, eps, x.grad)

    with torch.no_grad():
        adv_logits = model(imagenet_normalize(adv))  # (1, num_classes)
        adv_probs = torch.softmax(adv_logits, dim=1).squeeze(0)  # (num_classes,)
        top1_idx = int(adv_probs.argmax().item())
        top1_conf = float(adv_probs[top1_idx].item())
        conf_orig_class = float(adv_probs[orig_label].item())

    attack_results.append(
        {
            "epsilon": eps,
            "image": adv.cpu(),
            "top1_label": categories[top1_idx],
            "top1_conf": top1_conf,
            "conf_orig_class": conf_orig_class,
            "top1_idx": top1_idx,
        }
    )

In [ ]:
# Presentation-aligned dark theme (Reveal black.css style)
BG = "#111111"
FG = "#eeeeee"
plt.rcParams.update(
    {
        "figure.facecolor": BG,
        "axes.facecolor": BG,
        "axes.edgecolor": FG,
        "axes.labelcolor": FG,
        "text.color": FG,
        "xtick.color": FG,
        "ytick.color": FG,
        "axes.titlecolor": FG,
        "figure.titlesize": 12,
    }
)

n = len(attack_results)
fig, axes = plt.subplots(2, n, figsize=(2.2 * n, 5), gridspec_kw={"height_ratios": [4, 1]})
axes = np.asarray(axes)
if axes.ndim == 1:
    axes = axes.reshape(2, 1)  # matplotlib returns (2,) when n==1

for i, row in enumerate(attack_results):
    ax_img = axes[0, i]
    # imshow expects (H, W, 3) — permute from (3, H, W)
    disp = row["image"].squeeze(0).permute(1, 2, 0).numpy()
    ax_img.imshow(np.clip(disp, 0, 1))
    ax_img.axis("off")
    title = "Original" if row["epsilon"] == 0 else f"ε={row['epsilon']}"
    ax_img.set_title(title, color=FG, fontsize=10)

    ax_txt = axes[1, i]
    ax_txt.axis("off")
    lbl = row["top1_label"][:40] + ("…" if len(row["top1_label"]) > 40 else "")
    ax_txt.text(
        0.5,
        0.5,
        f"{lbl}\n{row['top1_conf']:.1%}",
        ha="center",
        va="center",
        fontsize=8,
        color=FG,
        wrap=True,
    )

fig.suptitle("FGSM: original vs perturbed — top-1 label and confidence", color=FG, y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
eps_x = [r["epsilon"] for r in attack_results]
conf_orig = [r["conf_orig_class"] for r in attack_results]
top1_match = [1.0 if r["top1_idx"] == orig_label else 0.0 for r in attack_results]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(eps_x, conf_orig, "o-", color="#4ecdc4", label="Confidence on original class")
ax.plot(eps_x, top1_match, "s--", color="#ff6b6b", label="Top-1 matches original (1=yes)")
ax.set_xlabel("ε (FGSM step size)")
ax.set_ylabel("Score")
ax.set_title("Robustness vs perturbation strength")
ax.legend(facecolor="#222222", edgecolor=FG, labelcolor=FG)
ax.grid(True, alpha=0.25, color=FG)
plt.tight_layout()
plt.show()

## Conclusion

**Key takeaway:** imperceptible changes (ε < 0.01) can already flip model predictions. Imagine this on a traffic sign classifier in a self-driving car.